In [ ]:
# NOTEBOOK NAME
# pyFLEXTRKRsandbox.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr
import pandas as pd

# from pathlib import Path      # used to play with pathnames to save

# from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

# mapping things
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader

# for adding lat/lon gridlines on plots
import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS AND ELEVATION AND PyFLEXTRKR FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/CustomFunctions')
from CustomFunctions1 import *
sys.path.insert(0, '/scratch/v46/sg3241/tmp/PyFLEXTRKR/')  # folder *containing* pyflextrkr/
import logging
logging.basicConfig(level=logging.INFO)
from pyflextrkr.idcells_reflectivity import idcells_reflectivity
from pyflextrkr.tracksingle_driver import tracksingle_driver
from pyflextrkr.gettracks import gettracknumbers
from pyflextrkr.trackstats_driver import trackstats_driver

# for adding a colourful topo base map to the CAPI plots
from custom_elevation import fetch_srtm, fetch_gebco_local
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

import glob

import shutil # I think this is to rename the output file from PyFLEXTRKR
import gc # something to prevent memory leaks

from pathlib import Path      # used to play with pathnames to save

from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

import matplotlib.colors as mcolors
import matplotlib.cm as cm

# from matplotlib.patches import Circle # for radar range ring circles on the map
# from matplotlib.lines import Line2D # for plotting stars on the map legend

In [ ]:
# RETRIEVE FEATURES THAT EXIST AT EACH TIME STEP

logging.basicConfig(level=logging.INFO)

# CHOOSE YOUR RADAR

# Radar Number Catalogue:
# Down The Coast YES Dual-Pol: 22 is Mackay,    106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marburg (near Bris)
# Down The Coast not Dual-Pol: 19 is Cairns,    8 is Gympie
# Down The Coast NO DOPPLER:   24 is Bowen,     23 is Gladstone (SPECIAL ELEVATION ANGLES)
# 0.8, 1.6, 2.4
# 3.6, 5.6
# 8.0, 11.5
# 16.0, 22.0, 32.0
#Inland
# Down inland YES Dual-Pol:    74 is Greenvale, 98 is Taroom,    108 is Towoomba
# Down inland not Dual-Pol:    78 is Weipa,     72 is Emerald
RadarIDno = '22' 

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 3
RadarDay   = 9
# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '00:00'
LoopEndTime   = '23:55'

# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD

# USER CHOICE FOLLOW-ON SECTION

# RADAR CHOICE FOLLOW-ON
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton (Brisbane)'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marburg'
elif (RadarIDno == '19'):
    RadarSiteName = 'Cairns'
elif (RadarIDno == '8'):
    RadarSiteName = 'Gympie'
elif (RadarIDno == '24'):
    RadarSiteName = 'Bowen'
elif (RadarIDno == '23'):
    RadarSiteName = 'Gladstone'
elif (RadarIDno == '74'):
    RadarSiteName = 'Greenvale'
elif (RadarIDno == '98'):
    RadarSiteName = 'Taroom'
elif (RadarIDno == '108'):
    RadarSiteName = 'Towoomba'
elif (RadarIDno == '78'):
    RadarSiteName = 'Weipa'
elif (RadarIDno == '72'):
    RadarSiteName = 'Emerald'
else:
    RadarSiteName = 'Site ' + RadarIDno

# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# Longitude shift correction for known coordinate errors
if RadarSiteName == 'Townsville':
    LonShift = 146.5505 - (-19.4195)
else:
    LonShift = 0.0

# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):

    houri = MinOfDay // 60
    mini  = MinOfDay % 60

    RadarFileTime      = str(houri).zfill(2) + str(mini).zfill(2) + '00'
    RadarFileTimePrint = RadarFileTime[0:2] + ':' + RadarFileTime[2:4] + ':' + RadarFileTime[4:6]

    print(f'Working on {RadarFileTimePrint}')

    RadarGridPath = (
        '/scratch/v46/sg3241/tmp/NetCDFs/CompressedRadarGrids/'
        + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
        + RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc'
    )

    # --- Check file exists without loading it into memory ---
    if not os.path.exists(RadarGridPath):
        print(f'File missing for {RadarFileTimePrint}, skipping: {RadarGridPath}')
        continue

    # --- Extract scheduled date/time from filename ---
    input_basename = os.path.basename(RadarGridPath)
    parts          = input_basename.replace('.nc', '').split('_')
    scheduled_date = parts[1]
    scheduled_time = parts[2]

    # --- Output path ---
    output_path = (
        '/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/SingleTimes/'
        + RadarIDno + '/' + YYYY + MM + DD + '/'
    )
    os.makedirs(output_path, exist_ok=True)

    # --- Skip if output already exists ---
    scheduled_outfile = output_path + f'cellidfile_{scheduled_date}_{scheduled_time}.nc'
    if os.path.exists(scheduled_outfile):
        print(f'Already exists, skipping')
        continue

    # --- Config ---
    pyflextrkr_config = {
        'input_format'          : 'netcdf',
        'reflectivity_varname'  : 'corrected_reflectivity',
        'time_dimname'          : 'time',
        'z_dimname'             : 'z',
        'y_dimname'             : 'y',
        'x_dimname'             : 'x',
        'x_coordname'           : 'x',
        'y_coordname'           : 'y',
        'z_coordname'           : 'z',
        'lon_coordname'         : 'lon',
        'lat_coordname'         : 'lat',
        'radar_lon_varname'     : 'radar_longitude',
        'radar_lat_varname'     : 'radar_latitude',
        'dx'                    : 1000.0,
        'dy'                    : 1000.0,
        'is_3d'                 : True,
        'z_coord_type'          : 'height',
        'sfc_dz_min'            : 500.0,
        'sfc_dz_max'            : 3000.0,
        'echotop_gap'           : 2,
        'default_sfc_height'    : 0.0,
        'radar_sensitivity'     : 0.0,
        'absConvThres'          : 40.0,
        'minZdiff'              : 8.0,
        'truncZconvThres'       : 42.0,
        'mindBZuse'             : 10.0,
        'dBZforMaxConvRadius'   : 45.0,
        'conv_rad_increment'    : 1.0,
        'conv_rad_start'        : 1.0,
        'bkg_refl_increment'    : 5.0,
        'maxConvRadius'         : 5.0,
        'radii_expand'          : [1, 2, 3, 4, 5],
        'weakEchoThres'         : 15.0,
        'bkgrndRadius'          : 11.0,
        'min_corearea'          : 4,
        'min_cellarea'          : 4,
        'tracking_outpath'      : output_path,
        'cloudid_filebase'      : 'cellidfile_tmp_',
        'fillval'               : -9999,
        'return_diag'           : False,
        'convolve_method'       : 'fft',
        'dilate_method'         : 'orig',
        'expand_method'         : 'orig',
        'echotop_method'        : 'orig',
        'remove_smallcores'     : True,
        'remove_smallcells'     : False,
    }

    # --- Run identification ---
    try:
        outfile = idcells_reflectivity(RadarGridPath, pyflextrkr_config)
        shutil.move(outfile, scheduled_outfile)
        print(f'Output written to: {scheduled_outfile}')
    except Exception as e:
        print(f'FAILED for {RadarFileTimePrint}: {e}')

    # --- Explicitly release memory after every timestep ---
    finally:
        gc.collect()


In [ ]:
FeaturesXR = xr.open_dataset('/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/SingleTimes/cellidfile_20241205_001500.nc')

In [ ]:
FeaturesXR

In [ ]:
# FUNNELING IN THE FEATURES FROM INDIVIDUAL TIME STEPS TO GET STATS
sys.path.insert(0, '/scratch/v46/sg3241/tmp/PyFLEXTRKR/')
logging.basicConfig(level=logging.WARNING)


# --- Paths ---
radar_id      = '22'
date_str      = '20240309'
cellid_path   = f'/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/SingleTimes/{radar_id}/{date_str}/'
tracking_path = f'/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Tracking/{radar_id}/{date_str}/'
stats_path    = f'/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/{radar_id}/{date_str}/'

os.makedirs(tracking_path, exist_ok=True)
os.makedirs(stats_path,    exist_ok=True)

# --- Get start/end times from available cellid files ---
cellid_files = sorted(glob.glob(os.path.join(cellid_path, 'cellidfile_*.nc')))
print(f"Found {len(cellid_files)} cellid files")

def datetime_from_cellid(filepath):
    basename = os.path.basename(filepath)           # cellidfile_20241205_120000.nc
    parts    = basename.replace('.nc', '').split('_')
    return pd.Timestamp(f"{parts[1]} {parts[2][:2]}:{parts[2][2:4]}:{parts[2][4:6]}")

t_start = datetime_from_cellid(cellid_files[0])
t_end   = datetime_from_cellid(cellid_files[-1])

# Convert to epoch seconds (required by tracksingle_driver)
epoch           = pd.Timestamp('1970-01-01')
start_basetime  = (t_start - epoch).total_seconds()
end_basetime    = (t_end   - epoch).total_seconds()

startdate_str   = t_start.strftime('%Y%m%d.%H%M%S')
enddate_str     = t_end.strftime('%Y%m%d.%H%M%S')

print(f"Time range: {t_start} to {t_end}")
print(f"Epoch seconds: {start_basetime} to {end_basetime}")

# --- Full config ---
config = {
    # --- Feature type ---
    'feature_type'                  : 'radar_cells',

    # --- Paths ---
    'tracking_outpath'              : cellid_path,    # where cellid files live
    'stats_outpath'                 : stats_path,

    # --- File bases ---
    'cloudid_filebase'              : 'cellidfile_',
    'singletrack_filebase'          : 'track_',
    'tracknumbers_filebase'         : 'tracknumbers_',
    'trackstats_filebase'           : 'trackstats_',
    'trackstats_sparse_filebase'    : 'trackstats_sparse_',
    'pixeltracking_filebase'        : 'celltracks_',

    # --- Time range ---
    'startdate'                     : startdate_str,
    'enddate'                       : enddate_str,
    'start_basetime'                : start_basetime,
    'end_basetime'                  : end_basetime,
    'time_format'                   : 'yyyymodd.hhmmss',
    'datatimeresolution'            : 5.0 / 60.0,     # hours (5-min data)

    # --- Grid parameters ---
    'dx'                            : 1000.0,          # metres
    'dy'                            : 1000.0,          # metres
    'pixel_radius'                  : 1.0,             # km
    'area_method'                   : 'fixed',

    # --- Variable names in cellid files ---
    'feature_varname'               : 'feature_number',
    'nfeature_varname'              : 'nfeatures',
    'featuresize_varname'           : 'npix_feature',
    'ref_varname'                   : 'dbz_comp',

    # --- Dimension names in cellid files ---
    'x_dimname'                     : 'lon',
    'y_dimname'                     : 'lat',
    'time_dimname'                  : 'time',

    # --- Tracking parameters ---
    'timegap'                       : 0.25,            # hours
    'othresh'                       : 0.3,             # 30% overlap
    'nmaxlinks'                     : 10,
    'maxnclouds'                    : 1000,
    'fillval'                       : -9999,

    # --- Track duration ---
    'duration_range'                : [2, 288],
    'duration_range_auto_update'    : True,
    'duration_range_round_base'     : 10,
    'remove_shorttracks'            : 1,

    # --- Output options ---
    'trackstats_dense_netcdf'       : 1,
    'match_pixel_dt_thresh'         : 60.0,            # seconds

    # --- Dimension names in output files ---
    'tracks_dimname'                : 'tracks',
    'times_dimname'                 : 'times',

    # --- Parallelisation (0=serial, 1=local dask cluster) ---
    'run_parallel'                  : 0,

    # --- No terrain file ---
    'terrain_file'                  : None,
    'rangemask_varname'             : None,
}

# ── Step 1: Pairwise linking ───────────────────────────────────────────────────
print("\n--- Step 1: tracksingle (pairwise linking) ---")
tracksingle_driver(config)
print("tracksingle complete.")

# ── Step 2: Assemble pairs into full tracks ────────────────────────────────────
print("\n--- Step 2: gettracknumbers (assemble full tracks) ---")
gettracknumbers(config)
print("gettracknumbers complete.")

# ── Step 3: Track statistics ───────────────────────────────────────────────────
print("\n--- Step 3: trackstats (compute statistics) ---")
trackstats_outfile = trackstats_driver(config)
print(f"trackstats complete.")
print(f"Output: {trackstats_outfile}")


In [ ]:
feature_number = FeaturesXR2['feature_number'].isel(time=0).values
npix_feature   = FeaturesXR2['npix_feature'].values
nfeatures      = int(FeaturesXR2['nfeatures'].values[0])

print(f"Number of features detected: {nfeatures}")
print(f"Pixels per feature: {npix_feature}")
print(f"Areas in km²: {npix_feature}  (1 pixel = 1 km²)")

# Quick plot of the feature label map
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(feature_number, origin='lower', cmap='tab20')
ax.set_title(f'Feature label map — {nfeatures} features detected')
plt.tight_layout()
plt.show()

In [ ]:
# test_path = '/scratch/v46/sg3241/tmp/NetCDFs/CompressedRadarGrids/22/2024/12/05/22_20241205_120000.nc'
# ds = xr.open_dataset(test_path)

test_path = '/scratch/v46/sg3241/tmp/NetCDFs/PyFLEXTRKR/Stats/22/20240309/trackstats_20240309.000000_20240309.235500.nc'
ds = xr.open_dataset(test_path)
print(ds)

In [ ]:
# FEATURE TRACKS PLOTTING

# CHOOSE YOUR RADAR

# Radar Number Catalogue:
# Down The Coast YES Dual-Pol: 22 is Mackay,    106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marburg (near Bris)
# Down The Coast not Dual-Pol: 19 is Cairns,    8 is Gympie
# Down The Coast NO DOPPLER:   24 is Bowen,     23 is Gladstone (SPECIAL ELEVATION ANGLES)
# 0.8, 1.6, 2.4
# 3.6, 5.6
# 8.0, 11.5
# 16.0, 22.0, 32.0
#Inland
# Down inland YES Dual-Pol:    74 is Greenvale, 98 is Taroom,    108 is Towoomba
# Down inland not Dual-Pol:    78 is Weipa,     72 is Emerald
RadarIDno = '22' 

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 3
RadarDay   = 9
# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '00:00'
LoopEndTime   = '23:55'

# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD

# USER CHOICE FOLLOW-ON SECTION

# RADAR CHOICE FOLLOW-ON
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton (Brisbane)'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marburg'
elif (RadarIDno == '19'):
    RadarSiteName = 'Cairns'
elif (RadarIDno == '8'):
    RadarSiteName = 'Gympie'
elif (RadarIDno == '24'):
    RadarSiteName = 'Bowen'
elif (RadarIDno == '23'):
    RadarSiteName = 'Gladstone'
elif (RadarIDno == '74'):
    RadarSiteName = 'Greenvale'
elif (RadarIDno == '98'):
    RadarSiteName = 'Taroom'
elif (RadarIDno == '108'):
    RadarSiteName = 'Towoomba'
elif (RadarIDno == '78'):
    RadarSiteName = 'Weipa'
elif (RadarIDno == '72'):
    RadarSiteName = 'Emerald'
else:
    RadarSiteName = 'Site ' + RadarIDno

# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# TOWNSVILLE LON NEEDS TO BE SHIFTED
# Longitude shift correction for known coordinate errors
if RadarSiteName == 'Townsville':
    LonShift = 146.5505 - (-19.4195)
else:
    LonShift = 0.0
    # ── User settings ──────────────────────────────────────────────────────────────
CoarsenLevel       = 1       # DEM coarsening factor
min_duration_steps = 0       # minimum track duration (timesteps) to plot
# ──────────────────────────────────────────────────────────────────────────────

RadarFileTime = '000000'

# LOAD IN AN EXAMPLE RADAR FILE TO GRAB LAT AND LON LIMITS FROM
RadarGridsFolder = 'CompressedRadarGrids'
NetCDFstoragePath = ('/scratch/v46/sg3241/tmp/NetCDFs/' + RadarGridsFolder + '/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/'
                                                         + RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')
# try to load in the netcdf file and if it doesn't work, just keep going through the loop
try:
    xgrid = xr.open_dataset(NetCDFstoragePath)
except FileNotFoundError:
    print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')

fig, ax = plt.subplots(figsize=(8, 6),
                       subplot_kw={'projection': ccrs.PlateCarree()})

# TERRAIN SHADING USING LOCAL GEBCO DEM
lon_min, lon_max = float(xgrid.lon.min()) + LonShift, float(xgrid.lon.max()) + LonShift
lat_min, lat_max = float(xgrid.lat.min()), float(xgrid.lat.max())

try:
    gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
    dem_da = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)
    if dem_da is None:
        raise ValueError('GEBCO DEM returned None')
    dem_da   = dem_da.coarsen(lon=CoarsenLevel, lat=CoarsenLevel, boundary='trim').mean()
    dem_lon  = dem_da.lon.values
    dem_lat  = dem_da.lat.values
    dem_data = dem_da.values
    if dem_lon.ndim == 1 and dem_lat.ndim == 1:
        dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
    else:
        dem_lon_2d, dem_lat_2d = dem_lon, dem_lat
    if not np.any(np.isfinite(dem_data)):
        raise ValueError('DEM has no finite values in this domain')

    colours = [
        '#dde4e8',
        '#c4dec2', '#e4edc9', '#f3f0cf',
        '#e9d7bd', '#ddc4aa', '#cfb194', '#b58f6e',
    ]
    bounds = [-1000.0, 0.0, 200.0, 400.0, 600.0, 800.0, 1000.0, 1200.0, 5000.0]
    cmap_elev = ListedColormap(colours)
    norm_elev = BoundaryNorm(bounds, len(colours), clip=True)
    ax.pcolormesh(dem_lon_2d, dem_lat_2d, dem_data,
                  cmap=cmap_elev, norm=norm_elev, alpha=0.8,
                  transform=ccrs.PlateCarree())
    ax.contour(dem_lon_2d, dem_lat_2d, dem_data,
               levels=[0.0], colors='black', linewidths=0.5,
               transform=ccrs.PlateCarree(), zorder=15)

        # Draw 0 m contour as an accurate coastline
    coast_contour = ax.contour(
        dem_lon_2d,
        dem_lat_2d,
        dem_data,
        levels=[0.0],
        colors='black',
        linewidths=0.5,
        transform=ccrs.PlateCarree(),
        zorder=15,  # above radar and topo
    )
    
    # Draw 400 m contour
    coast_contour = ax.contour(
        dem_lon_2d,
        dem_lat_2d,
        dem_data,
        levels=[400.0],
        colors='black',
        linewidths=0.3,
        transform=ccrs.PlateCarree(),
        zorder=15,  # above radar and topo
    )

except Exception as e:
    print(f'Terrain shading failed: {e}')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.30, zorder=1)
    ax.add_feature(cfeature.LAND,  facecolor='#E8E8E8',   alpha=0.30, zorder=2)

# ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)


# plot a star for the location of the radar on the map
ax.plot(float(xgrid.radar_longitude) + LonShift, float(xgrid.radar_latitude),
        marker='*', color='black', markersize=8, transform=ccrs.PlateCarree(), zorder=20)
ax.plot(float(xgrid.radar_longitude) + LonShift, float(xgrid.radar_latitude),
        marker='*', color='white', markersize=4, transform=ccrs.PlateCarree(), zorder=21)

# ── GRIDLINES ─────────────────────────────────────────────────────────────────

# Add MINOR gridlines (tenth degrees) - thin
gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
gl_minor.xlocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees
gl_minor.ylocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees

# Add MID LEVEL gridlines (half degrees) - standard width with labels
gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
gl_mid.xlocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees
gl_mid.ylocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees

# Add MAJOR gridlines (full degrees) - thick with labels
gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
gl_major.xlocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
gl_major.ylocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees

# Format labels
gl_mid.xformatter = LONGITUDE_FORMATTER
gl_mid.yformatter = LATITUDE_FORMATTER

gl_major.xformatter = LONGITUDE_FORMATTER
gl_major.yformatter = LATITUDE_FORMATTER

# Remove labels from top and right
gl_mid.top_labels = False
gl_mid.right_labels = False
gl_mid.bottom_labels = True
gl_mid.left_labels = True

gl_major.top_labels = False
gl_major.right_labels = False
gl_major.bottom_labels = True
gl_major.left_labels = True

# ── TRACK PLOTTING ────────────────────────────────────────────────────────────
# Colourmap: time of day mapped to 0–1 (midnight=0, midnight=1)
cmap_tracks = cm.nipy_spectral
norm_tracks = mcolors.Normalize(vmin=0, vmax=24)   # hours of day

n_tracks_plotted = 0

for i in range(len(ds['track_duration'])):
    duration = int(ds['track_duration'].values[i])

    # Skip very short tracks
    if duration < min_duration_steps:
        continue

    # Extract valid timesteps only
    lats = ds['cell_meanlat'].values[i, :duration]
    lons = ds['cell_meanlon'].values[i, :duration]
    times = ds['base_time'].values[i, :duration]

    # Skip if all positions are NaN
    if np.all(np.isnan(lats)) or np.all(np.isnan(lons)):
        continue

    # Convert times to hour-of-day (0–24) for colouring
    times_pd    = pd.DatetimeIndex(times)
    hours_of_day = times_pd.hour + times_pd.minute / 60.0 + times_pd.second / 3600.0

    # Plot each segment individually so colour varies along the track
    for j in range(len(lons) - 1):
        if np.isnan(lons[j]) or np.isnan(lats[j]):
            continue
        if np.isnan(lons[j+1]) or np.isnan(lats[j+1]):
            continue
        colour = cmap_tracks(norm_tracks(hours_of_day[j]))
        ax.plot(
            [lons[j], lons[j+1]],
            [lats[j], lats[j+1]],
            color=colour,
            linewidth=0.8,
            alpha=0.7,
            transform=ccrs.PlateCarree(),
            zorder=20,
        )

    n_tracks_plotted += 1

# ── COLOURBAR ─────────────────────────────────────────────────────────────────
sm = cm.ScalarMappable(cmap=cmap_tracks, norm=norm_tracks)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, orientation='vertical',
                    pad=0.02, shrink=0.7, aspect=25)
cbar.set_label('Time of day (UTC hour)', fontsize=9)
cbar.set_ticks([0, 3, 6, 9, 12, 15, 18, 21, 24])
cbar.set_ticklabels(['00:00', '03:00', '06:00', '09:00',
                     '12:00', '15:00', '18:00', '21:00', '24:00'])

# ── EXTENT AND TITLE ──────────────────────────────────────────────────────────
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax.set_title('Feature Tracks for ' + RadarSiteName + ' Radar\non ' + \
              RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' UTC')
plt.tight_layout()

# ── save ──────────────────────────────────────────────────────────────────────
SaveFolder = ('/scratch/v46/sg3241/tmp/pngImages/FeatureTracks/' +
              RadarIDno + '/' + RadarFileDate + '/')
SaveFile = RadarIDno + '_' + RadarFileDate + '_FeatureTracks_PyFLEXTRKR.png'

SavePath   = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi=300)
plt.close()
print(f"Saved to {SavePath}")